# 00 準備：環境とデータの場所を確認する

この研究のコードは、次の順で使います。

1. **デモ（合成データ）**で全ステップを実行し、`06_verify` で参照値と一致することを確かめる（AI が作った解析・画面・検証の動作確認）
2. **実データ**の場所を `pbrtqc.local.json` に書き、同じノートブックを上から実行する（人が再現する）
3. `06_verify` で、以前の解析結果（参照）と数字が一致するかを確かめる（検証）

実データは**読むだけ**で、書き換えません。仮名化したキャッシュと集計表は `work/`（デモは `work_demo/`）にでき、どちらも Git には入りません。
ノートブックに表示するのは集計表だけです。患者ごとの行は表示しないでください。

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "pbrtqc").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
pd.set_option("display.max_columns", 30, "display.width", 200)
from pbrtqc import config as C, pipeline
print(C.describe())

> **いまのモード**は上のセルの `profile=` で確認してください（`demo` = 合成データ、`real` = 実データ）。モードを変えたら、VS Code の「Restart」でカーネルを再起動してから上から実行し直します。

## 1. Python とパッケージの版を確認する
同じ版なら、同じデータから同じ数字が出ます。違う版だと最後の桁が変わることがあります。

In [ ]:
import platform
from importlib import metadata
want = dict(l.split("==") for l in (ROOT / "requirements.txt").read_text().splitlines() if "==" in l and not l.startswith("#"))
print("Python", platform.python_version(), "(公開時 3.14)")
for name, v in want.items():
    have = metadata.version(name)
    print(f"{name:12s} {have:10s} {'OK' if have == v else '← 公開時は ' + v}")

## 2. 実データの場所を設定する（実データを使うときだけ）
下のセルのパスを自分の環境に合わせて書き換え、`WRITE = True` にして実行します。`pbrtqc.local.json` ができます（Git には入りません）。
実行したらカーネルを再起動してください。デモに戻すときは `"profile": "demo"` にします。

In [ ]:
local = {
    "profile": "real",
    "lis_raw_dir": str(Path.home() / "pbrtqc/data/raw/lis_2023_2025"),                  # LIS の出力（*.xlsx）が入ったフォルダ
    "qc_raw_file": str(Path.home() / "pbrtqc/data/qc/再検データ抽出QC2023_2025.xlsx"),   # IQC の出力ファイル
    "reference_tables": str(Path.home() / "pbrtqc/outputs/tables"),                     # 照合に使う以前の結果表
}
WRITE = False
if WRITE:
    (ROOT / "pbrtqc.local.json").write_text(json.dumps(local, ensure_ascii=False, indent=1), encoding="utf-8")
    print("pbrtqc.local.json を作りました。カーネルを再起動してください。")

## 3. 入力ファイルがあるかを確認する（中身は読みません）

In [ ]:
files = sorted(C.LIS_RAW.glob("*.xlsx")) + sorted(C.LIS_RAW.glob("*.csv"))
print("LIS:", len(files), "ファイル")
for p in files:
    print(f"  {p.name}  {p.stat().st_size / 1e6:.1f} MB")
print("IQC:", C.QC_RAW.name, "あり" if C.QC_RAW.exists() else "なし（デモなら 01 で作られます）")

## 4. 実行するステップの一覧

In [ ]:
pd.DataFrame([(g, s, d) for s, g, _, _, d in pipeline.steps()], columns=["グループ", "ステップ", "内容"])